# Notebook 3: ElasticNet
Objetivo: `km_media_prox28`. 6 corridas de ElasticNet: mini-rejilla de 3 valores de alpha x 2 valores de `l1_ratio`.
El baseline NO se vuelve a registrar: ya existe en el Notebook 1. Aquí solo se recalculan sus métricas para comparar.

In [1]:
import sys
sys.path.append("../src")
import itertools
import pandas as pd
from sklearn.linear_model import ElasticNet
import modelos_comunes as mc

mc.configurar_mlflow()

MLflow: http://3.18.4.17:5000 | experimento: modelo_km_dia


## 1. Datos

In [2]:
train, test = mc.cargar_datos()
print("train:", train.shape, "| test:", test.shape)
print("version datos:", mc.version_datos())

train: (8906, 29) | test: (1138, 29)
version datos: 72d353ac4a7d5867e3637ea3cfea39c6


## 2. Métricas del baseline (solo cálculo, sin registrar otra corrida)

In [3]:
base = mc.calcular_metricas(test[mc.OBJETIVO].values, test["km_media28"].values)
print(f"Baseline -> MAE {base['mae']:.2f} km/dia | MAE fecha {base['mae_fecha_dias']:.2f} dias")

Baseline -> MAE 16.56 km/dia | MAE fecha 2.47 dias


## 3. ElasticNet: alpha x l1_ratio
`l1_ratio` cerca de 0 se parece a Ridge; cerca de 1 se parece a Lasso. En Ridge los mejores resultados estuvieron en alpha pequeño, por eso la rejilla se concentra en valores bajos.

In [4]:
alphas = [0.001, 0.01, 0.1]
l1_ratios = [0.2, 0.8]
resultados = []
for a, l1 in itertools.product(alphas, l1_ratios):
    r = mc.correr_modelo("elasticnet", f"elasticnet_alpha_{a}_l1_{l1}",
                         ElasticNet(alpha=a, l1_ratio=l1, max_iter=20000),
                         {"alpha": a, "l1_ratio": l1}, train, test, base)
    resultados.append(r)

2026/11/23 21:15:17 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run elasticnet_alpha_0.001_l1_0.2 at: http://3.18.4.17:5000/#/experiments/2/runs/2b5ac08fcf6449a7bf5e6a0a086e7fc2
🧪 View experiment at: http://3.18.4.17:5000/#/experiments/2
elasticnet_alpha_0.001_l1_0.2 MAE 19.01 | CV 24.15 | mejora -14.8% | fecha 2.82 d


2026/11/23 21:15:34 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run elasticnet_alpha_0.001_l1_0.8 at: http://3.18.4.17:5000/#/experiments/2/runs/5c5f60820dec48fb92272d37f6e634ec
🧪 View experiment at: http://3.18.4.17:5000/#/experiments/2
elasticnet_alpha_0.001_l1_0.8 MAE 19.21 | CV 24.37 | mejora -16.0% | fecha 2.85 d


2026/11/23 21:15:51 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run elasticnet_alpha_0.01_l1_0.2 at: http://3.18.4.17:5000/#/experiments/2/runs/e7b2167d3cc54e27ae80ad1890061d88
🧪 View experiment at: http://3.18.4.17:5000/#/experiments/2
elasticnet_alpha_0.01_l1_0.2 MAE 17.73 | CV 23.57 | mejora -7.0% | fecha 2.57 d


... (2 outputs left out)

2026/11/23 21:16:25 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run elasticnet_alpha_0.1_l1_0.2 at: http://3.18.4.17:5000/#/experiments/2/runs/65adfe7a3d2f446e82f3cb418e5d8a75
🧪 View experiment at: http://3.18.4.17:5000/#/experiments/2
elasticnet_alpha_0.1_l1_0.2  MAE 17.55 | CV 22.70 | mejora -6.0% | fecha 2.53 d


2026/11/23 21:16:42 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run elasticnet_alpha_0.1_l1_0.8 at: http://3.18.4.17:5000/#/experiments/2/runs/07b510fb9723451da16f4dcc962b60d7
🧪 View experiment at: http://3.18.4.17:5000/#/experiments/2
elasticnet_alpha_0.1_l1_0.8  MAE 17.01 | CV 23.24 | mejora -2.7% | fecha 2.40 d


## 4. Resumen

In [5]:
tabla = pd.DataFrame(resultados)[["run_name", "alpha", "l1_ratio", "cv_mae", "mae", "rmse",
                                  "mejora_mae_pct", "mae_fecha_dias", "mejora_fecha_pct"]]
tabla.sort_values("cv_mae").round(3)

,run_name,alpha,l1_ratio,cv_mae,mae,rmse,mejora_mae_pct,mae_fecha_dias,mejora_fecha_pct
4,elasticnet_alpha_0.1_l1_0.2,0.100,0.2,22.697,17.548,22.312,-5.978,2.531,-2.433
5,elasticnet_alpha_0.1_l1_0.8,0.100,0.8,23.244,17.006,22.118,-2.706,2.400,2.880
2,elasticnet_alpha_0.01_l1_0.2,0.010,0.2,23.574,17.726,22.733,-7.050,2.572,-4.089
0,elasticnet_alpha_0.001_l1_0.2,0.001,0.2,24.148,19.006,24.125,-14.784,2.823,-14.232
3,elasticnet_alpha_0.01_l1_0.8,0.010,0.8,24.150,18.626,23.661,-12.485,2.746,-11.121
1,elasticnet_alpha_0.001_l1_0.8,0.001,0.8,24.372,19.210,24.378,-16.012,2.855,-15.528
